In [22]:
from langgraph.graph import END, START, StateGraph
from typing import TypedDict 
from  langgraph.types import Send, interrupt, Command
import subprocess
from openai import OpenAI
import textwrap
from langchain.chat_models import init_chat_model
from typing_extensions import Annotated
import operator
import base64
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

llm = init_chat_model("openai:gpt-4o-mini")

class State(TypedDict):
    video_file : str
    audio_file : str
    transcription : str
    summaries : Annotated[list[str], operator.add] 
    thumbnail_prompts : Annotated[list[str], operator.add] 
    thumbnail_sketched : Annotated[list[str], operator.add] 
    finally_summary : str
    user_feedback : str  
    chosen_prompt : str # 선택된 썸네일 프롬포트 

In [23]:
# 노드 생성 

# 오디오 추출 (FFmpeg를 사용해서)  
def extract_audio(state : State):
    output_file = state["video_file"].replace("mp4","mp3") # mp4 -> mp3로 변환
    #ffmpeg - 이걸 이렇게 해주는 이유는 openai는 변환 작업에 대해 분당 요금을 받기 때문이다.  또한 오디오 파일의 속도를 빠르게 해서 변환해도 퀄리티가 떨어지지않음 (이로인해 변환비용 감소 )
    command = [
        "ffmpeg",
        "-i", 
        state["video_file"],
        "-filter:a",# 수정작업 
        "atempo=2.0" , #속도 임 2배 설정 
        "-y",  # ffmpeg에게 해당 파일을 덮어씌울 것인지 물어보면 yes라고 대답하도록 하는 코드  
        output_file
    ]
    subprocess.run(command)
    return{
        "audio_file" : output_file
    }

# 오디오 파일 변환 
def transcribe_audio(state : State):
    # use audio file 
    client = OpenAI()
    with open(state["audio_file"], "rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model = "whisper-1",
            response_format= "text",
            file = audio_file,
            language= "en", # 모델에게 영상이 어떤 언어로 되어있는지 알려주는것 , 모델에게 주는 힌트 
            prompt="Netherlands, Rotterdam, Amsterdam, The Hage", # gpt 프롬포트 같은게 아닌 모델에 단어 목록 같은 걸 주는 것임. (d이걸 주지 않으면 모델이 저 사람이 뭐라고 말하는지 알아내는게 힘들 수도있음, 모델이 듣게될 단어에 대한 힌트)
        )
        return{
            "transcription" : transcription
        }

def dispatch_summarizers(state : State):
    transcription = state["transcription"]
    chunks = []
    for i, chunk in enumerate(textwrap.wrap(transcription, 500)):
        chunks.append({"id" : i+1, "chunk" : chunk}) # 전사본의 chunk를 분리해서 id와 함께 리스트 생성 
    return [Send("summarize_chunk", chunk) for chunk in chunks ] # summerize_chunk노드를 보내고 chunks에 있는 모든 chunk에 대해 실행
      # 여기서 여러개의 send 커맨드를 분배 하고 

def summarize_chunk(chunk):   # id 값과 함께 이 chunk를 요약할 것임. 
    chunk_id = chunk["id"]
    chunk = chunk["chunk"]

    response = llm.invoke(
        f"""
        Please summarize the following text.
        
        Text : {chunk}
        """
    )
    summary = f"[Chunk {chunk_id}] {response.content}"
    return {"summaries" : [summary]}  # 요약본이 들어올 때마다 summaries 에 추가된다. 
   #  print(f"Summarizing chunk id :{chunk_id} chunk : {chunk[:100]}\n\n ==== \n\n") # 이렇게 찍힌 값을 ai에 넣어서 요약하는것

# 최종 요약본함수  
def mega_summary(state :State):
    all_summaries = "\n".join(state["summaries"]) # 모든 요약본을 하나의 string으로 합치는것 ! 각 요약본에 줄바꿈으로 개행문자
    prompt = f"""
        You are given multiple summaries of different chunks from a video transcription.

        Please create a comprehensive final summary that combines all the key points.

        Individual summaries:

        {all_summaries}
    """# 너는 한 영상에서 나온 텍스트의 여러 청크들로 만든 요약본을 받을 거야 . 모든 핵심 포인트를 결합한 종합 요약본을 작성해줘 

    response = llm.invoke(prompt)

    return {
        "finally_summary" : response.content
    }

def dispatch_artists(state: State):
    return [
        Send(
            "generate_thumbnails"
            ,{
                "id": i,
                "summary": state["finally_summary"],
            },
        )
        for i in [1,2,3,4,5]  # generate_thumbnails 함수 5번 수행 
    ] 

def generate_thumbnails(args):
    concept_id = args["id"]
    summary = args["summary"]

    prompt = f"""
    Based on this video summary, create a detailed visual prompt for a YouTube thumbnail.

    Create a detailed prompt for generating a thumbnail image that would attract viewers. Include:
        - Main visual elements
        - Color scheme
        - Text overlay suggestions
        - Overall composition
    
    Summary: {summary}
    """

    response = llm.invoke(prompt)

    thumbnmail_prompt = response.content

    client = OpenAI()

    result = client.images.generate(
        model = "gpt-image-1",
        prompt = thumbnmail_prompt,
        quality="low" , # 돈도 덜 들고 빨리끝남
        moderation="low",
        size="auto",
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)  # base64 이미지를 bytes 로 변환
    filename = f"thumbnai_{concept_id}.jpg"

    with open(filename, "wb") as file:
        file.write(image_bytes) 
    return {"thumbnail_prompts" : [thumbnmail_prompt], "thumbnail_sketched" : [filename]}

def human_feedback(state :State):
    answer = interrupt({
        "chosen_thumbnail" : "Which thumbnail do you like the most?", #원하는 썸네일 번호 
        "feedback" : "Provide any feedback or changes you'd like for the final thumbnail."
    })
    user_feedback = answer["user_feedback"]
    chosen_prompt = answer["chosen_prompt"]
    return {
        "user_feedback" : user_feedback,
        "chosen_prompt" : state["thumbnail_prompts"][chosen_prompt -1 ]
    }

def generate_hd_thumbnail(state : State):
    chosen_prompt = state["chosen_prompt"] 
    user_feedback = state["user_feedback"]

    # ai 모델한테 선택된 프롬포트와 유저 피드백을 통합한 프롬포트를 만들어 달라고 한다. 
    prompt = f"""
    You are a professional YouTube thumbnail designer. Take this original thumbnail prompt and create an enhanced version that incorporates the user's specific feedback.

    ORIGINAL PROMPT:
    {chosen_prompt}

    USER FEEDBACK TO INCORPORATE:
    {user_feedback}

    Create an enhanced prompt that:
        1. Maintains the core concept from the original prompt
        2. Specifically addresses and implements the user's feedback requests
        3. Adds professional YouTube thumbnail specifications:
            - High contrast and bold visual elements
            - Clear focal points that draw the eye
            - Professional lighting and composition
            - Optimal text placement and readability with generous padding from edges
            - Colors that pop and grab attention
            - Elements that work well at small thumbnail sizes
            - IMPORTANT: Always ensure adequate white space/padding between any text and the image borders
    """
    prompt = f"""
    You are a professional YouTube thumbnail designer. Take this original thumbnail prompt and create an enhanced version that incorporates the user's specific feedback.

    ORIGINAL PROMPT:
    {chosen_prompt}

    USER FEEDBACK TO INCORPORATE:
    {user_feedback}

    Create an enhanced prompt that:
        1. Maintains the core concept from the original prompt
        2. Specifically addresses and implements the user's feedback requests
        3. Adds professional YouTube thumbnail specifications:
            - High contrast and bold visual elements
            - Clear focal points that draw the eye
            - Professional lighting and composition
            - Optimal text placement and readability with generous padding from edges
            - Colors that pop and grab attention
            - Elements that work well at small thumbnail sizes
            - IMPORTANT: Always ensure adequate white space/padding between any text and the image borders
    """

    response = llm.invoke(prompt)

    final_thumbnail_prompt = response.content

    client = OpenAI()

    result = client.images.generate(
        model="gpt-image-1.5",
        prompt=final_thumbnail_prompt,
        quality="high", # 퀄리티 높게 
        moderation="low",
        size="auto",
    )

    image_bytes = base64.b64decode(result.data[0].b64_json)

    with open("thumbnail_final.jpg", "wb") as file:
        file.write(image_bytes)

In [24]:
# 그래프 생성 

graph_builder = StateGraph(State)

graph_builder.add_node("extract_audio", extract_audio)
graph_builder.add_node("transcribe_audio", transcribe_audio)
graph_builder.add_node("summarize_chunk", summarize_chunk)
graph_builder.add_node("mega_summary", mega_summary)
graph_builder.add_node("generate_thumbnails", generate_thumbnails)
graph_builder.add_node("human_feedback", human_feedback)
graph_builder.add_node("generate_hd_thumbnail", generate_hd_thumbnail)

graph_builder.add_edge(START, "extract_audio")
graph_builder.add_edge("extract_audio", "transcribe_audio")
graph_builder.add_conditional_edges(
    "transcribe_audio", dispatch_summarizers, ["summarize_chunk"]
)
graph_builder.add_edge("summarize_chunk", "mega_summary")
graph_builder.add_conditional_edges("mega_summary", dispatch_artists, ["generate_thumbnails"]) # mega_summary에는 dispatch_artist함수가 있고 generate_thumbnails노드를 온디맨드로 실행
graph_builder.add_edge("generate_thumbnails", "human_feedback")
graph_builder.add_edge("human_feedback", "generate_hd_thumbnail")
graph_builder.add_edge("generate_hd_thumbnail", END)

graph = graph_builder.compile(checkpointer= memory)

In [25]:
config = {
    "configurable" : {"thread_id" : "1"}
}

In [26]:
graph.invoke({"video_file": "netherlands.mp4"}, config= config)

{'video_file': 'netherlands.mp4',
 'audio_file': 'netherlands.mp3',
 'transcription': "This episode is supported by Audible. Amazon Prime members can get an incredible 66% off their first 3 months at audible.com slash SUIBHNE Only available for a limited time, so sign up fast. The Moravian and Riviera-born Lowlands were once inhabited by the Holy Roman Empire. They were well-known for building dams, canals and windmills to pump water into the land. And for creating one of Europe's trade centres in Broodmaan. The early Dutch were famous for trading, shipbuilding and hydro-engineering. And the French language that was spoken there, began to evolve into the Dutch language. Or, the language of the Lowlands. And so, the Dutch were born. Feudalism helped sway in the land, and as the trade increased, the cities of Amsterdam, Brugge and Antwerpen prospered. And soon, these trading posts became more and more integrated into the Hanseatic League, a sort of trade agreement in the North Sea after 

In [27]:
#위의 결과의 intrupt 를 더 자세하게 (참고고)

snapshot = graph.get_state(config)
snapshot.interrupts 

# snapshot.next  #다음 실행 할 도구  

(Interrupt(value={'chosen_thumbnail': 'Which thumbnail do you like the most?', 'feedback': "Provide any feedback or changes you'd like for the final thumbnail."}, id='a519aa0855e4fccfd17608032e37bd84'),)

In [28]:
response = {
    "user_feedback": "Make sure the fella is smiling, remove any mention of audible, or any logo, and give it a photo realistic, 3d style.",
    "chosen_prompt": 4,
}

graph.invoke(
    Command(resume=response),
    config=config,
)

{'video_file': 'netherlands.mp4',
 'audio_file': 'netherlands.mp3',
 'transcription': "This episode is supported by Audible. Amazon Prime members can get an incredible 66% off their first 3 months at audible.com slash SUIBHNE Only available for a limited time, so sign up fast. The Moravian and Riviera-born Lowlands were once inhabited by the Holy Roman Empire. They were well-known for building dams, canals and windmills to pump water into the land. And for creating one of Europe's trade centres in Broodmaan. The early Dutch were famous for trading, shipbuilding and hydro-engineering. And the French language that was spoken there, began to evolve into the Dutch language. Or, the language of the Lowlands. And so, the Dutch were born. Feudalism helped sway in the land, and as the trade increased, the cities of Amsterdam, Brugge and Antwerpen prospered. And soon, these trading posts became more and more integrated into the Hanseatic League, a sort of trade agreement in the North Sea after 